# VGG-11 / VGG-13 / VGG-16 on CIFAR-10
## With and Without Batch Normalization

**Research objective:** Study the effect of CNN depth, repeated 3×3 convolutions, and Batch Normalization on CIFAR-10 classification.

### Models
- VGG-11
- VGG-11 + BN
- VGG-13
- VGG-13 + BN
- VGG-16
- VGG-16 + BN

### What this notebook measures
- Test accuracy
- Training/validation loss and accuracy
- Number of trainable parameters
- Training time
- Best validation accuracy
- Test confusion matrix
- Per-class accuracy
- Accuracy vs. parameter count
- Depth vs. accuracy
- BN vs. no-BN
- Analytical comparison of stacked 3×3 convolutions

> **Important:** The original VGG paper targets large-scale ImageNet. This notebook adapts the VGG convolutional design to CIFAR-10's 32×32 images rather than copying the ImageNet classifier unchanged.


## 1. Research background

The VGG paper investigates increasing network depth using very small **3×3 convolution filters**, reporting strong results with 16–19 weight layers.

For CIFAR-10, images are only **32×32×3**, so the network is implemented with the VGG block pattern but a CIFAR-10-sized classifier.

**Key research questions**

1. Does increasing depth from VGG-11 → VGG-13 → VGG-16 improve accuracy?
2. Does Batch Normalization improve optimization/convergence?
3. How does parameter count change with depth?
4. Why are multiple 3×3 convolutions attractive compared with one larger convolution?
5. Is additional depth worth the additional computation/parameters on CIFAR-10?


In [ ]:
# ============================================================
# 2. Install / import dependencies
# ============================================================
!pip -q install -U torch torchvision torchaudio scikit-learn pandas matplotlib seaborn tqdm

import os
import time
import copy
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 3. Reproducibility + experiment configuration
# ============================================================
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility can reduce raw GPU speed slightly.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

# 30 epochs is a practical default for a Colab T4.
# Increase to 50–100 for a stronger final research run if time permits.
EPOCHS = 30

LEARNING_RATE = 0.05
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

VAL_SIZE = 5000
DATA_DIR = "./data"
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)


## 4. CIFAR-10 dataset

CIFAR-10 contains 10 classes of 32×32 RGB images.

For training we use:
- Random crop with padding
- Random horizontal flip
- CIFAR-10 channel normalization

Validation and test data use only tensor conversion + normalization.

The same split and preprocessing are used for every VGG model so the comparison remains fair.


In [ ]:
# ============================================================
# 5. Dataset and DataLoaders
# ============================================================
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

full_train_aug = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=train_transform
)

full_train_eval = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=eval_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=eval_transform
)

# Fixed split indices so every model sees exactly the same train/validation samples.
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_train_aug), generator=generator).tolist()

val_indices = indices[:VAL_SIZE]
train_indices = indices[VAL_SIZE:]

train_dataset = torch.utils.data.Subset(full_train_aug, train_indices)
val_dataset = torch.utils.data.Subset(full_train_eval, val_indices)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

classes = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", classes)


In [ ]:
# ============================================================
# 6. Visualize CIFAR-10 examples
# ============================================================
def denormalize(img):
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    return img * std + mean

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for ax, img, label in zip(axes.flat, images[:15], labels[:15]):
    img = denormalize(img).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[label.item()])
    ax.axis("off")

plt.suptitle("CIFAR-10 Training Examples")
plt.tight_layout()
plt.show()


## 7. VGG architecture

### VGG configurations used

| Model | Convolutional configuration |
|---|---|
| VGG-11 | 1, 1, 2, 2, 2 convolution layers per block |
| VGG-13 | 2, 2, 2, 2, 2 convolution layers per block |
| VGG-16 | 2, 2, 3, 3, 3 convolution layers per block |

Each convolution uses:
- 3×3 kernel
- stride 1
- padding 1
- ReLU
- optional BatchNorm

Each block ends with 2×2 max pooling.

The classifier is adapted for CIFAR-10. Instead of the very large ImageNet VGG classifier, this implementation uses **Adaptive Average Pooling + a compact linear classifier**, which is much more appropriate for 32×32 images.


In [ ]:
# ============================================================
# 8. VGG configuration + model implementation
# ============================================================
VGG_CONFIGS = {
    "VGG11": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG13": [64, 64, "M", 128, 128, "M",
              256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG16": [64, 64, "M", 128, 128, "M",
              256, 256, 256, "M", 512, 512, 512, "M",
              512, 512, 512, "M"],
}

class VGG_CIFAR(nn.Module):
    def __init__(self, config, num_classes=10, batch_norm=False, dropout=0.0):
        super().__init__()

        layers = []
        in_channels = 3

        for item in config:
            if item == "M":
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                out_channels = item

                layers.append(
                    nn.Conv2d(
                        in_channels,
                        out_channels,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=not batch_norm
                    )
                )

                if batch_norm:
                    layers.append(nn.BatchNorm2d(out_channels))

                layers.append(nn.ReLU(inplace=True))
                in_channels = out_channels

        self.features = nn.Sequential(*layers)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode="fan_out", nonlinearity="relu"
                )
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


def build_model(model_name, batch_norm=False):
    return VGG_CIFAR(
        VGG_CONFIGS[model_name],
        num_classes=NUM_CLASSES,
        batch_norm=batch_norm,
        dropout=0.0
    )


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_summary_row(model_name, batch_norm):
    model = build_model(model_name, batch_norm)
    params = count_parameters(model)

    return {
        "Model": model_name + (" + BN" if batch_norm else ""),
        "Depth": int(model_name.replace("VGG", "")),
        "BatchNorm": batch_norm,
        "Parameters": params,
        "Parameters_M": params / 1e6
    }

architecture_df = pd.DataFrame([
    model_summary_row("VGG11", False),
    model_summary_row("VGG11", True),
    model_summary_row("VGG13", False),
    model_summary_row("VGG13", True),
    model_summary_row("VGG16", False),
    model_summary_row("VGG16", True),
])

architecture_df


In [ ]:
# ============================================================
# 9. Verify tensor shapes and print one model
# ============================================================
model = build_model("VGG16", batch_norm=True).to(DEVICE)

print(model)

dummy = torch.randn(2, 3, 32, 32).to(DEVICE)
with torch.no_grad():
    output = model(dummy)

print("\nInput shape :", tuple(dummy.shape))
print("Output shape:", tuple(output.shape))
print("Parameters  :", f"{count_parameters(model):,}")


## 10. Loss, optimizer and scheduler

For a fair comparison, every model uses the same training recipe:

- Cross-Entropy Loss
- SGD
- Momentum = 0.9
- Weight decay = 5e-4
- Cosine Annealing learning-rate schedule

The BN and non-BN models differ only in the presence of Batch Normalization.


In [ ]:
# ============================================================
# 11. Training and evaluation functions
# ============================================================
criterion = nn.CrossEntropyLoss()

def accuracy_from_logits(logits, targets):
    predictions = logits.argmax(dim=1)
    return (predictions == targets).sum().item()


def train_one_epoch(model, loader, optimizer, scaler=None):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += accuracy_from_logits(outputs, labels)
        total += batch_size

    return running_loss / total, 100.0 * running_correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += accuracy_from_logits(outputs, labels)
        total += batch_size

    return running_loss / total, 100.0 * running_correct / total


def fit_model(model_name, batch_norm, epochs=EPOCHS):
    seed_everything(SEED)

    model = build_model(model_name, batch_norm).to(DEVICE)

    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        nesterov=True
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    scaler = (
        torch.amp.GradScaler("cuda")
        if DEVICE.type == "cuda"
        else None
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": []
    }

    best_val_acc = -1.0
    best_state = None
    best_epoch = 0

    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        epoch_start = time.perf_counter()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scaler
        )
        val_loss, val_acc = evaluate(model, val_loader)

        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr_now)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        elapsed = time.perf_counter() - epoch_start

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:6.2f}% | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:6.2f}% | "
            f"LR {lr_now:.6f} | "
            f"{elapsed:.1f}s"
        )

    total_time = time.perf_counter() - start_time

    # Restore best validation checkpoint
    model.load_state_dict(best_state)

    # Save checkpoint
    suffix = "bn" if batch_norm else "no_bn"
    checkpoint_path = os.path.join(
        CHECKPOINT_DIR, f"{model_name}_{suffix}.pth"
    )

    torch.save({
        "model_name": model_name,
        "batch_norm": batch_norm,
        "state_dict": model.state_dict(),
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "history": history
    }, checkpoint_path)

    result = {
        "Model": model_name + (" + BN" if batch_norm else ""),
        "Architecture": model_name,
        "Depth": int(model_name.replace("VGG", "")),
        "BatchNorm": batch_norm,
        "Parameters": count_parameters(model),
        "Parameters_M": count_parameters(model) / 1e6,
        "Best_Val_Acc": best_val_acc,
        "Best_Epoch": best_epoch,
        "Training_Time_Min": total_time / 60,
        "History": history,
        "Checkpoint": checkpoint_path
    }

    return model, result


## 12. Quick sanity check

Before starting all six experiments, train one model for only two epochs.

If this cell runs correctly, set `RUN_ALL_EXPERIMENTS = True` below.


In [ ]:
# ============================================================
# 12A. Sanity check: VGG11 without BN for 2 epochs
# ============================================================
SANITY_CHECK = False

if SANITY_CHECK:
    _, sanity_result = fit_model("VGG11", False, epochs=2)
    print(sanity_result["Best_Val_Acc"])
else:
    print("Sanity check skipped. Set SANITY_CHECK=True if needed.")


# 13. Run all six VGG experiments

**This is the main compute cell.**

Expected runtime depends on the current Colab T4 availability, PyTorch version, and dataloader speed. If you have limited time, first run 10 epochs to verify the full pipeline, then change `RUN_EPOCHS` to 30–50 for the final experiment.


In [ ]:
# ============================================================
# 13. MAIN EXPERIMENT
# ============================================================
RUN_ALL_EXPERIMENTS = True

# Use 30 for the intended run.
# You can temporarily set this to 5 or 10 for a pipeline test.
RUN_EPOCHS = EPOCHS

experiment_specs = [
    ("VGG11", False),
    ("VGG11", True),
    ("VGG13", False),
    ("VGG13", True),
    ("VGG16", False),
    ("VGG16", True),
]

all_results = []
trained_models = {}

if RUN_ALL_EXPERIMENTS:
    total_experiments = len(experiment_specs)

    for idx, (model_name, batch_norm) in enumerate(experiment_specs, start=1):
        print("\n" + "=" * 80)
        print(
            f"EXPERIMENT {idx}/{total_experiments}: "
            f"{model_name} | BatchNorm={batch_norm}"
        )
        print("=" * 80)

        model, result = fit_model(
            model_name,
            batch_norm,
            epochs=RUN_EPOCHS
        )

        trained_models[result["Model"]] = model
        all_results.append(result)

        # Free unused cached GPU memory between experiments
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\nAll experiments completed.")
else:
    print("RUN_ALL_EXPERIMENTS=False. No models trained.")


In [ ]:
# ============================================================
# 14. Convert experiment results into a comparison table
# ============================================================
if all_results:
    results_df = pd.DataFrame([
        {
            "Model": r["Model"],
            "Depth": r["Depth"],
            "BatchNorm": r["BatchNorm"],
            "Parameters_M": r["Parameters_M"],
            "Best_Val_Acc_%": r["Best_Val_Acc"],
            "Best_Epoch": r["Best_Epoch"],
            "Training_Time_Min": r["Training_Time_Min"]
        }
        for r in all_results
    ])

    results_df = results_df.sort_values(
        ["Depth", "BatchNorm"]
    ).reset_index(drop=True)

    display(results_df.style.format({
        "Parameters_M": "{:.3f}",
        "Best_Val_Acc_%": "{:.2f}",
        "Training_Time_Min": "{:.2f}"
    }))
else:
    print("No results yet.")


## 15. Test-set evaluation

Only the **best validation checkpoint** for each model is evaluated on the test set. This avoids selecting the model using the test set itself.


In [ ]:
# ============================================================
# 15. Evaluate all best checkpoints on the test set
# ============================================================
@torch.no_grad()
def predict_all(model, loader):
    model.eval()

    all_targets = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_predictions.extend(predictions)
        all_targets.extend(labels.numpy())

    return np.array(all_targets), np.array(all_predictions)


test_records = []

for result in all_results:
    model_key = result["Model"]
    model = trained_models[model_key]

    y_true, y_pred = predict_all(model, test_loader)

    test_acc = 100.0 * np.mean(y_true == y_pred)

    test_records.append({
        "Model": model_key,
        "Depth": result["Depth"],
        "BatchNorm": result["BatchNorm"],
        "Parameters_M": result["Parameters_M"],
        "Best_Val_Acc_%": result["Best_Val_Acc"],
        "Test_Acc_%": test_acc,
        "Best_Epoch": result["Best_Epoch"],
        "Training_Time_Min": result["Training_Time_Min"],
        "y_true": y_true,
        "y_pred": y_pred
    })

test_df = pd.DataFrame(test_records)

display(test_df[[
    "Model", "Depth", "BatchNorm",
    "Parameters_M", "Best_Val_Acc_%",
    "Test_Acc_%", "Best_Epoch", "Training_Time_Min"
]].style.format({
    "Parameters_M": "{:.3f}",
    "Best_Val_Acc_%": "{:.2f}",
    "Test_Acc_%": "{:.2f}",
    "Training_Time_Min": "{:.2f}"
}))


## 16. Training curves

These plots help distinguish:
- underfitting
- overfitting
- faster convergence
- instability
- the effect of BatchNorm


In [ ]:
# ============================================================
# 16. Plot training/validation curves
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

for result in all_results:
    label = result["Model"]
    h = result["History"]
    epochs_axis = range(1, len(h["train_loss"]) + 1)

    axes[0].plot(epochs_axis, h["train_loss"], label=f"{label} Train")
    axes[0].plot(
        epochs_axis, h["val_loss"],
        linestyle="--",
        label=f"{label} Val"
    )

axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend(ncol=2, fontsize=8)

for result in all_results:
    label = result["Model"]
    h = result["History"]
    epochs_axis = range(1, len(h["train_acc"]) + 1)

    axes[1].plot(
        epochs_axis, h["train_acc"],
        label=f"{label} Train"
    )
    axes[1].plot(
        epochs_axis, h["val_acc"],
        linestyle="--",
        label=f"{label} Val"
    )

axes[1].set_title("Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


## 17. Depth vs. accuracy

This is the central experiment from the VGG research idea: compare networks of increasing depth while keeping the basic design principle fixed.


In [ ]:
# ============================================================
# 17. Depth vs accuracy
# ============================================================
plt.figure(figsize=(10, 6))

for bn_value, group in test_df.groupby("BatchNorm"):
    label = "With BatchNorm" if bn_value else "Without BatchNorm"

    plt.plot(
        group["Depth"],
        group["Test_Acc_%"],
        marker="o",
        linewidth=2,
        label=label
    )

    for _, row in group.iterrows():
        plt.annotate(
            f'{row["Test_Acc_%"]:.2f}%',
            (row["Depth"], row["Test_Acc_%"]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center"
        )

plt.xticks([11, 13, 16])
plt.xlabel("VGG Depth")
plt.ylabel("Test Accuracy (%)")
plt.title("Effect of Network Depth on CIFAR-10 Test Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 18. Parameter count vs. accuracy

A deeper network usually requires more parameters. This plot checks whether the extra parameters are accompanied by improved test performance.


In [ ]:
# ============================================================
# 18. Parameters vs accuracy
# ============================================================
plt.figure(figsize=(10, 6))

for _, row in test_df.iterrows():
    plt.scatter(
        row["Parameters_M"],
        row["Test_Acc_%"],
        s=100
    )
    plt.annotate(
        row["Model"],
        (row["Parameters_M"], row["Test_Acc_%"]),
        xytext=(6, 6),
        textcoords="offset points"
    )

plt.xlabel("Trainable Parameters (Millions)")
plt.ylabel("Test Accuracy (%)")
plt.title("Parameter Count vs. CIFAR-10 Test Accuracy")
plt.grid(True, alpha=0.3)
plt.show()


## 19. BatchNorm effect

For each depth, compare the BN and non-BN model.

A useful descriptive quantity is:

**BN accuracy gain = Accuracy(BN) − Accuracy(no BN)**

Do not assume the gain must be positive; report what your experiment actually measures.


In [ ]:
# ============================================================
# 19. BN vs no-BN comparison
# ============================================================
pivot = test_df.pivot(
    index="Depth",
    columns="BatchNorm",
    values="Test_Acc_%"
).rename(columns={False: "No_BN", True: "BN"})

pivot["BN_Gain_pp"] = pivot["BN"] - pivot["No_BN"]

display(pivot.style.format("{:.2f}"))

plt.figure(figsize=(9, 5))
x = np.arange(len(pivot.index))
width = 0.35

plt.bar(x - width/2, pivot["No_BN"], width, label="Without BN")
plt.bar(x + width/2, pivot["BN"], width, label="With BN")

plt.xticks(x, pivot.index.astype(str))
plt.xlabel("VGG Depth")
plt.ylabel("Test Accuracy (%)")
plt.title("Effect of Batch Normalization")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()


## 20. Confusion matrices

For each model, inspect which CIFAR-10 classes are confused most often.


In [ ]:
# ============================================================
# 20. Confusion matrices for all models
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax, record in zip(axes.flat, test_records):
    cm = confusion_matrix(record["y_true"], record["y_pred"])

    sns.heatmap(
        cm,
        annot=False,
        fmt="d",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=ax
    )

    ax.set_title(
        f'{record["Model"]}\nTest Acc: {record["Test_Acc_%"]:.2f}%'
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()


## 21. Per-class accuracy

This identifies whether a model performs differently across the 10 CIFAR-10 classes.


In [ ]:
# ============================================================
# 21. Per-class accuracy for each model
# ============================================================
per_class_rows = []

for record in test_records:
    y_true = record["y_true"]
    y_pred = record["y_pred"]

    for class_id, class_name in enumerate(classes):
        mask = y_true == class_id

        class_acc = (
            100.0 * np.mean(y_pred[mask] == class_id)
            if mask.sum() > 0 else np.nan
        )

        per_class_rows.append({
            "Model": record["Model"],
            "Class": class_name,
            "Accuracy_%": class_acc
        })

per_class_df = pd.DataFrame(per_class_rows)

display(
    per_class_df.pivot(
        index="Class",
        columns="Model",
        values="Accuracy_%"
    ).style.format("{:.2f}")
)


# 22. 3×3 stacking analysis

### Why stack 3×3 convolutions?

Consider a convolution with `C` input and output channels.

A single 5×5 convolution has approximately:

**25C² parameters**

Two 3×3 convolutions have approximately:

**2 × 9C² = 18C² parameters**

So the two 3×3 layers use:

**18 / 25 = 72%**

of the convolutional kernel parameters of one 5×5 layer, while producing an effective receptive field of 5×5.

There is also an additional ReLU between the two convolutions, giving the network more nonlinear transformations.

This is one of the core architectural ideas behind VGG.


In [ ]:
# ============================================================
# 22A. Analytical 3x3 vs 5x5 parameter comparison
# ============================================================
C_values = [32, 64, 128, 256, 512]

rows = []

for C in C_values:
    one_5x5 = 25 * C * C
    two_3x3 = 2 * 9 * C * C

    rows.append({
        "Channels": C,
        "One_5x5_Params": one_5x5,
        "Two_3x3_Params": two_3x3,
        "Two_3x3 / One_5x5": two_3x3 / one_5x5,
        "Parameter_Saving_%": 100 * (1 - two_3x3 / one_5x5)
    })

kernel_df = pd.DataFrame(rows)

display(kernel_df.style.format({
    "One_5x5_Params": "{:,}",
    "Two_3x3_Params": "{:,}",
    "Two_3x3 / One_5x5": "{:.2f}",
    "Parameter_Saving_%": "{:.2f}"
}))


## 23. Receptive-field calculation

Ignoring pooling for this simple demonstration:

- one 3×3 convolution → receptive field = 3×3
- two stacked 3×3 convolutions → receptive field = 5×5
- three stacked 3×3 convolutions → receptive field = 7×7

General rule for stride-1 convolutions:

**r_new = r_old + (k − 1)**

For k = 3:

**r_new = r_old + 2**


In [ ]:
# ============================================================
# 23A. Receptive field demonstration
# ============================================================
receptive_field = 1

rows = []

for layer in range(1, 5):
    receptive_field += 3 - 1
    rows.append({
        "Number_of_3x3_layers": layer,
        "Effective_Receptive_Field": f"{receptive_field}x{receptive_field}"
    })

pd.DataFrame(rows)


# 24. Final experimental summary

This cell produces the main table you can use in your report/presentation.

**Do not manually type accuracy values into the report.** Copy the actual values produced by your completed run.


In [ ]:
# ============================================================
# 24. Final summary table
# ============================================================
final_columns = [
    "Model",
    "Depth",
    "BatchNorm",
    "Parameters_M",
    "Best_Val_Acc_%",
    "Test_Acc_%",
    "Best_Epoch",
    "Training_Time_Min"
]

final_table = test_df[final_columns].copy()

final_table["Parameters_M"] = final_table["Parameters_M"].round(3)
final_table["Best_Val_Acc_%"] = final_table["Best_Val_Acc_%"].round(2)
final_table["Test_Acc_%"] = final_table["Test_Acc_%"].round(2)
final_table["Training_Time_Min"] = final_table["Training_Time_Min"].round(2)

display(final_table)


## 25. Automatically generated observations

The following statements are generated from the measured results, so they do not assume that a particular model will win before training.


In [ ]:
# ============================================================
# 25. Automatic observations
# ============================================================
if len(test_df) > 0:

    best_row = test_df.loc[test_df["Test_Acc_%"].idxmax()]

    print("===== AUTOMATIC EXPERIMENT OBSERVATIONS =====\n")

    print(
        f"Highest measured test accuracy: {best_row['Model']} "
        f"with {best_row['Test_Acc_%']:.2f}%."
    )

    print(
        f"Parameter count: {best_row['Parameters_M']:.3f} million."
    )

    print(
        f"Best validation accuracy: {best_row['Best_Val_Acc_%']:.2f}% "
        f"at epoch {int(best_row['Best_Epoch'])}."
    )

    print("\nDepth comparison:")
    for depth in [11, 13, 16]:
        subset = test_df[test_df["Depth"] == depth]
        if len(subset) == 2:
            no_bn = subset.loc[subset["BatchNorm"] == False, "Test_Acc_%"].iloc[0]
            bn = subset.loc[subset["BatchNorm"] == True, "Test_Acc_%"].iloc[0]
            print(
                f"VGG-{depth}: No-BN={no_bn:.2f}%, "
                f"BN={bn:.2f}%, BN difference={bn-no_bn:+.2f} percentage points"
            )

    print("\nImportant: these observations describe this run on this split and "
          "training configuration; they are not universal claims about VGG.")


# 26. Save results

The notebook exports:
- final comparison CSV
- per-class accuracy CSV
- architecture information CSV
- individual model checkpoints

These files can be used in your report or GitHub project.


In [ ]:
# ============================================================
# 26. Export results
# ============================================================
OUTPUT_DIR = "./vgg_cifar10_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

final_table.to_csv(
    os.path.join(OUTPUT_DIR, "final_vgg_comparison.csv"),
    index=False
)

per_class_df.to_csv(
    os.path.join(OUTPUT_DIR, "per_class_accuracy.csv"),
    index=False
)

architecture_df.to_csv(
    os.path.join(OUTPUT_DIR, "architecture_parameters.csv"),
    index=False
)

# Save the full experiment history in a JSON-friendly format
history_export = {}

for result in all_results:
    history_export[result["Model"]] = result["History"]

with open(
    os.path.join(OUTPUT_DIR, "training_history.json"),
    "w"
) as f:
    json.dump(history_export, f, indent=2)

print("Saved files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)


# 27. Report-ready conclusion template

After running the notebook, replace the bracketed values using the generated table.

### Conclusion

This project implemented VGG-11, VGG-13 and VGG-16 architectures on CIFAR-10, each with and without Batch Normalization. The experiment was designed to isolate the effect of network depth and Batch Normalization under the same data split and training protocol.

The measured test accuracies were:

- VGG-11: **[insert result]**
- VGG-11 + BN: **[insert result]**
- VGG-13: **[insert result]**
- VGG-13 + BN: **[insert result]**
- VGG-16: **[insert result]**
- VGG-16 + BN: **[insert result]**

The results should be interpreted together with parameter count and training time rather than accuracy alone. The stacked 3×3 design provides a larger effective receptive field through multiple small convolutions while introducing additional nonlinear transformations. Analytically, two 3×3 convolutions require fewer kernel parameters than one 5×5 convolution when the channel width is held constant.

Batch Normalization was evaluated as an experimental factor rather than assumed to be beneficial. Its measured effect should be reported using the actual difference observed in the six-model experiment.

### Limitations

1. CIFAR-10 is substantially smaller and simpler than ImageNet.
2. This notebook uses a CIFAR-10-adapted VGG classifier rather than reproducing the original ImageNet training setup.
3. Results depend on random seed, augmentation, optimizer, learning-rate schedule and number of epochs.
4. A longer training budget may change the relative performance of the variants.

### Final research takeaway

The experiment demonstrates how controlled architectural comparisons can be used to study the relationship among **depth, parameter count, Batch Normalization, optimization behavior and classification accuracy**.


# 28. References

1. Simonyan, K. and Zisserman, A. **Very Deep Convolutional Networks for Large-Scale Image Recognition.** arXiv:1409.1556.
2. PyTorch / TorchVision documentation for VGG model definitions.
3. TorchVision documentation for CIFAR-10.
